# Faculty AI — Paper Autograding Lab

**Draft a rubric-based grade for a stack of student papers — grounded in the course material you actually taught.**

Point this notebook at your course material, your rubric, and a folder of student papers. For each paper it scores every rubric criterion, backs each score with quotes from the paper, computes a weighted overall grade, and drafts **two sets of notes** — one for you (why it landed where it did) and one you can hand back to the student.

It is a sibling of the **Class Participation Scoring Lab** and reuses the exact same stack — Amazon Bedrock + **Nova Lite**, `NovaMultimodalEmbeddings`, **FAISS**, and **LangChain**. The key idea it borrows: *"correct" and "on-topic" need a reference for what was taught* — so your course material is embedded, and each paper is judged against it, not against the model's own knowledge.

## What this notebook does

1. **Part 1** — install + connect to Bedrock.
2. **Part 2** — load your **course material** and build a searchable index (the ground truth for "did they use the concepts correctly").
3. **Part 3** — load your **rubric** (a structured, editable file) and the **folder of papers**, and reconcile against a roster so non-submitters surface.
4. **Part 4** — grade every paper: per-criterion tier + evidence + a weighted overall grade.
5. **Part 5** — **review and adjust** every grade (the important part), with a quick fairness check.
6. **Part 6** — export a dated report `.csv`/`.md` and per-student feedback files.
7. **Part 7** — where this does **not** belong.

> ⚠️ **Read this first.** This produces a **draft you review**, not a grade the tool assigns on its own. Student papers are student records — handle them under your institution's policy (FERPA in the US or your local equivalent). And this is **not** a plagiarism or AI-writing detector: never use its output in an academic-integrity decision. See Part 7.

# Part 1 — Setup *(~2 min)*

### 1.1 Install dependencies

In [1]:
%%capture
%pip install -r requirements.txt

### 1.2 Import tools and pick where the models run

This lab runs the same code two ways. Flip **`PROVIDER`**:

- **`"bedrock"`** — Amazon Bedrock + Nova (the SageMaker Studio path).
- **`"ollama"`** — models running **locally on your own machine** via [Ollama](https://ollama.com). No AWS account, nothing leaves your laptop. First: install Ollama, run `ollama serve`, and pull the two models — `ollama pull qwen2.5:7b` and `ollama pull nomic-embed-text`.

Everything downstream (indexing, grading, review, export) is identical either way.

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from mlu_utils.paper_tools import (
    load_rubric, load_papers, load_roster, reconcile_roster,
    PaperGrader, build_report, build_detail, write_student_feedback,
)

PROVIDER = "bedrock"        # <- "bedrock" (AWS / SageMaker) or "ollama" (local, no AWS)

if PROVIDER == "bedrock":
    import boto3
    from mlu_utils.embeddings import NovaMultimodalEmbeddings
    REGION = "us-east-1"
    bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)
    embeddings = NovaMultimodalEmbeddings(client=bedrock_runtime)
    grader_llm = None        # PaperGrader builds Nova Lite itself
    print("Using Amazon Bedrock (Nova) in", REGION)

elif PROVIDER == "ollama":
    from langchain_ollama import ChatOllama, OllamaEmbeddings
    CHAT_MODEL  = "qwen2.5:7b"        # try qwen2.5:14b if your machine can hold it
    EMBED_MODEL = "nomic-embed-text"
    embeddings = OllamaEmbeddings(model=EMBED_MODEL)
    grader_llm = ChatOllama(
        model=CHAT_MODEL,
        temperature=0.2,
        num_ctx=8192,        # CRITICAL: Ollama defaults to 2048, which truncates the paper + rubric
        num_predict=3000,    # room for the full JSON grade
        format="json",       # force valid JSON out of the grader
    )
    bedrock_runtime = None
    print(f"Using Ollama locally \u2014 chat={CHAT_MODEL}, embed={EMBED_MODEL} (is `ollama serve` running?)")

else:
    raise ValueError("PROVIDER must be 'ollama' or 'bedrock'")

Using Ollama locally — chat=qwen2.5:7b, embed=nomic-embed-text (is `ollama serve` running?)


# Part 2 — Load the day's course material *(~1 min)*

This is the **ground truth** for what was taught. When the grader judges whether a paper *used the course concepts correctly*, it judges only against what you load here — not against the model's own knowledge of the subject. Use the reading, the lecture notes, the chapter, the slides exported to PDF — whatever defines "the material" for this assignment.

### 2.1 🟢 EDIT ME — point to the course material

In [3]:
LESSON_PDF = "data/persona2_cs_data_structures.pdf"  # <- change to your assignment's material

### 2.2 Build the searchable course index

Loads the PDF, splits it into passages, embeds them with Nova Multimodal Embeddings, and indexes them in FAISS. `retriever` is what the grader queries to ground each paper's grade.

In [4]:
pages = PyPDFLoader(LESSON_PDF).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(pages)

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"Indexed {len(chunks)} passages from {len(pages)} pages of course material.")

Indexed 61 passages from 31 pages of course material.


# Part 3 — Load the rubric, the papers, and the roster *(~1 min)*

### 3.1 🟢 EDIT ME — your rubric, papers folder, and roster

- **`RUBRIC_PATH`** — a structured rubric file. Open **`data/sample_rubric.py`** and edit it in place: it's plain Python (criteria, weights, and 0–4 descriptors), readable top to bottom. Swap the whole thing for your assignment's rubric.
- **`PAPERS_DIR`** — a folder with one file per student. Filenames become student names (`Aisha_Rahman.pdf` → "Aisha Rahman"). `.pdf`, `.docx`, `.txt`, and `.md` all parse.
- **`ROSTER_PATH`** *(optional)* — a CSV with a `student_name` column. With a roster, enrolled students with **no paper** are flagged **missing**, and a paper whose name matches nobody is flagged **off-roster** (misnamed file). Without one, only students who submitted are visible.

In [5]:
RUBRIC_PATH  = "data/sample_rubric.py"       # <- your rubric (edit the file itself)
PAPERS_DIR   = "data/sample_papers"          # <- folder of student papers
ROSTER_PATH  = "data/sample_roster.csv"      # <- set to None to grade only what was submitted

### 3.2 What was submitted, and by whom

In [6]:
rubric      = load_rubric(RUBRIC_PATH)
submissions = load_papers(PAPERS_DIR)
roster      = load_roster(ROSTER_PATH) if ROSTER_PATH else None

matched, missing, off_roster = reconcile_roster(submissions, roster)

print(f"Rubric: {len(rubric.criteria)} criteria "
      f"({', '.join(c.name + ' ' + format(rubric.normalized_weights()[c.name]*100, '.0f') + '%' for c in rubric.criteria)})")
print(f"\nPapers to grade : {len(matched)}")
for name, sub in matched:
    print(f"  • {name:20s} {sub.words:5d} words   ({sub.path.split('/')[-1]})")
if missing:
    print(f"\nMissing (on roster, no paper): {', '.join(missing)}")
if off_roster:
    print(f"\n⚠️  Off-roster papers (name matches nobody — check the filename): "
          f"{', '.join(s.student_name for s in off_roster)}")

Rubric: 4 criteria (Thesis & Argument 25%, Use of Course Concepts 35%, Evidence & Support 25%, Clarity & Structure 15%)

Papers to grade : 3
  • Aisha Rahman           399 words   (Aisha_Rahman.txt)
  • Marcus Lee             228 words   (Marcus_Lee.txt)
  • Priya Chandra          228 words   (Priya_Chandra.txt)

Missing (on roster, no paper): Diego Alvarez


# Part 4 — Grade the papers *(~1–2 min for the sample)*

One grounded model call per paper scores **every criterion at once**, so the model reasons over the whole paper coherently. The **overall grade is computed in code** from the tiers and your rubric weights — the model never does the arithmetic. Missing papers are handled in code and never sent to the model.

In [7]:
grader = PaperGrader(retriever, bedrock_runtime, rubric, llm=grader_llm)
graded = grader.grade_all(matched, missing)

report_md, csv_rows = build_report(graded, rubric)
from IPython.display import Markdown, display
display(Markdown(report_md))

| Student | Overall | Grade | Thesis & Argument | Use of Course Concepts | Evidence & Support | Clarity & Structure | Confidence |
|---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| Aisha Rahman | 87.5% | B+ | 4 | 3 | 4 | 3 | high |
| Marcus Lee | 28.7% | F | 1 | 1 | 1 | 2 | low |
| Priya Chandra | 13.8% | F | 1 | 0 | 0 | 2 | low |
| Diego Alvarez | — | missing | — | — | — | — | high |

### 4.1 The reasoning behind each grade

The per-student breakdown: the note **for you** (private rationale), each criterion's tier and evidence quotes, and the draft note **for the student**. This is the part to actually read.

In [8]:
display(Markdown(build_detail(graded, rubric)))

### Aisha Rahman — 87.5% (B+) · confidence high
**For the professor:** The paper effectively argues its thesis and uses course concepts correctly, with strong evidence. The structure is good but could be improved for clarity.
- **Thesis & Argument** — 4/4. Clear thesis with a specific argument supported by relevant examples. _Evidence:_ "When a program's core operation is 'have I seen this before?', a hash table is the right data structure, because it trades a small, bounded amount of extra memory for average-case constant-time lookup that an array cannot match."; "The claim is narrow and I want to keep it narrow: for the single operation of membership on unordered data at scale, the hash table's average O(1) is decisive."
- **Use of Course Concepts** — 3/4. Uses course concepts correctly with minor imprecision in some details. _Evidence:_ "The array is simple and cache-friendly, but the cost of membership grows with the data, which is exactly the wrong direction for a deduplication task."; "For ordinary input, the average case is what you get, and it is the reason a hash table is the standard tool for a set."
- **Evidence & Support** — 4/4. Every major claim is supported with concrete examples or correct analysis. _Evidence:_ "With an unsorted array, checking whether an ID has appeared before is a linear scan: O(n) per check, and O(n^2) across the whole stream."; "The hash table spends memory to keep the load factor low, and its worst case degrades to O(n) if every key collides into one bucket — the pathological case where a bad hash function turns the table back into a linked list."
- **Clarity & Structure** — 3/4. Well organized; occasional awkward passage. _Evidence:_ "Choosing the Right Structure: Why Hash Tables Beat Arrays for Membership Tests"; "Conclusion: the right structure follows from the operation you run most."
**Draft note to student:** You've done a great job on this paper! Keep up the clear argumentation and precise use of concepts. Just work on making your transitions smoother to enhance readability.

### Marcus Lee — 28.7% (F) · confidence low
**For the professor:** The grade reflects the lack of focus on course concepts and the general nature of the comparison. Ensure the next draft includes specific examples from the material.
- **Thesis & Argument** — 1/4. The paper lacks a clear central claim and wanders between discussing linked lists and arrays. _Evidence:_ —
- **Use of Course Concepts** — 1/4. The paper does not engage with course concepts and instead provides a general comparison. _Evidence:_ —
- **Evidence & Support** — 1/4. The paper lacks concrete evidence or analysis, relying on general statements. _Evidence:_ —
- **Clarity & Structure** — 2/4. The paper is understandable but lacks a clear structure and organization. _Evidence:_ —
**Draft note to student:** Your paper could benefit from a clearer thesis and more precise engagement with the course content. Focus on one main argument and support it with concrete evidence.

### Priya Chandra — 13.8% (F) · confidence low
**For the professor:** This paper scores low on the use of course concepts and evidence, but shows basic clarity in personal reflections. It needs more engagement with the material.
- **Thesis & Argument** — 1/4. The paper lacks a clear central claim and instead provides personal reflections. _Evidence:_ "Overall I am enjoying the class and I think I am learning a lot."
- **Use of Course Concepts** — 0/4. The paper does not engage with the course concepts at all. _Evidence:_ —
- **Evidence & Support** — 0/4. The paper lacks any evidence or support for its claims. _Evidence:_ —
- **Clarity & Structure** — 2/4. The writing is understandable but lacks a clear structure, with some repetition. _Evidence:_ "Data structures is my favorite class though because the professor is really nice and explains things clearly."
**Draft note to student:** Your writing is clear, but you could improve by connecting your experiences to specific data structures and their applications. Try including examples from class to support your points.

### Diego Alvarez — missing
**For the professor:** On the roster but no paper was found. Confirm before entering a 0.


# Part 5 — Review and adjust *(the important part)*

The model drafts; **you decide.** Read the evidence in 4.1 and override anything that's wrong — a misread quote, a criterion the rubric weights differently than the model assumed, a paper that did something clever the rubric didn't anticipate. Transcription/parse artifacts and a student's clever move should never cost points silently.

### 5.1 🟢 EDIT ME — override any grade

Map a student name to the tiers you want per criterion. Only the criteria you list change; the overall grade recomputes from your tiers. Leave the dict empty to accept every draft as-is.

In [9]:
OVERRIDES = {
    # "Marcus Lee": {"Use of Course Concepts": 2, "Evidence & Support": 2},
}

for g in graded:
    if g.name in OVERRIDES:
        for c in g.criteria:
            if c.name in OVERRIDES[g.name]:
                c.tier = OVERRIDES[g.name][c.name]
        g.overall_percent = rubric.overall_percent(g.tier_map())
        g.letter = rubric.letter(g.overall_percent)
        g.confidence = "instructor-set"

graded.sort(key=lambda g: (g.overall_percent if g.overall_percent is not None else -1), reverse=True)
report_md, csv_rows = build_report(graded, rubric)
display(Markdown(report_md))

| Student | Overall | Grade | Thesis & Argument | Use of Course Concepts | Evidence & Support | Clarity & Structure | Confidence |
|---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| Aisha Rahman | 87.5% | B+ | 4 | 3 | 4 | 3 | high |
| Marcus Lee | 28.7% | F | 1 | 1 | 1 | 2 | low |
| Priya Chandra | 13.8% | F | 1 | 0 | 0 | 2 | low |
| Diego Alvarez | — | missing | — | — | — | — | high |

### 5.2 A 30-second fairness check

Before you export, sanity-check the distribution and the low-confidence grades — the ones most worth a human second look.

- Are the **low-confidence** grades the thin/ambiguous papers, or is the model unsure about something you can see clearly?
- Is any criterion doing **all** the work (every grade rides on one criterion)? That may mean the rubric weights need a rethink.
- Did anything land as **⚠️ unreadable** (a scanned PDF with no text layer) or **⚠️ re-run** (malformed model output)? Those need a hand, not a number.

In [10]:
from collections import Counter
dist = Counter(g.letter for g in graded if g.status == "graded")
print("Grade distribution:", dict(sorted(dist.items())))

low = [g.name for g in graded if g.status == "graded" and g.confidence.lower() == "low"]
print("Low-confidence (look here first):", ", ".join(low) or "none")

flags = [f"{g.name} ({g.status})" for g in graded if g.status in ("unreadable", "parse_error")]
print("Needs a human:", ", ".join(flags) or "none")

Grade distribution: {'B+': 1, 'F': 2}
Low-confidence (look here first): Marcus Lee, Priya Chandra
Needs a human: none


# Part 6 — Export your records *(~30 sec)*

Writes three things, all timestamped:
- a **`.csv`** — one row per student, one column per criterion, for your gradebook;
- a **`.md`** — the full report + reasoning, for your own records;
- a **`feedback/`** folder — one markdown file per student, containing **only the student-facing notes** (never your private rationale), ready to hand back.

In [11]:
from datetime import datetime
import csv, os

stamp = datetime.now().strftime("%Y%m%d_%H%M")
out_dir = f"grades_{stamp}"
os.makedirs(out_dir, exist_ok=True)

csv_path = os.path.join(out_dir, "grades.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(csv_rows[0].keys()))
    writer.writeheader()
    writer.writerows(csv_rows)

md_path = os.path.join(out_dir, "grades.md")
with open(md_path, "w", encoding="utf-8") as f:
    f.write(f"# Draft grades — {stamp}\n\n{report_md}\n\n---\n\n{build_detail(graded, rubric)}")

feedback_paths = write_student_feedback(graded, rubric, os.path.join(out_dir, "feedback"))

print(f"Wrote:\n  {csv_path}\n  {md_path}\n  {len(feedback_paths)} student feedback files in {out_dir}/feedback/")

Wrote:
  grades_20260731_1150/grades.csv
  grades_20260731_1150/grades.md
  3 student feedback files in grades_20260731_1150/feedback/


# Part 7 — Where this does **not** belong

This tool reads a student's paper and puts a number on it. Treat the number as the *start* of your judgment, not the end.

- **The grade is a draft.** The notebook is built so you review the evidence and adjust every grade before it counts. Never wire the raw output straight into a gradebook.
- **It is not a plagiarism or AI-writing detector.** It has no idea who wrote the paper or whether it's original. Do not use it — or its confidence score — in an academic-integrity case.
- **Grounded means grounded in *your* material.** If your course material is thin or off-target, the "used the concepts correctly" judgment is only as good as what you loaded in Part 2. A paper that goes beyond the material is treated as neutral, not wrong — but check the ones that surprised you.
- **Bias and voice.** Automated scoring can penalize multilingual writers, non-standard-but-valid structure, and unconventional arguments. Use the fairness check, and read the low-confidence and low-scoring papers yourself.
- **Consent and policy.** Student papers are protected records. Keep them, and anything this notebook exports, inside your institution's approved systems.